In [ ]:
import sys
import numpy as np, pandas as pd

# 1\ Load data
dataset = sys.argv[1]
data_path = 'data/{}/data_raw.tsv'.format(dataset) 
save_path = 'data/{}/data.tsv'.format(dataset)
data = pd.read_csv(data_path, index_col=0, sep='\t') # [n_gene, n_cell]
cells = data.columns.values
genes = data.index.values
print('Before filtering: number of genes is {}, number of cells is {}'.format(len(genes), len(cells)))

# 2\ Pre-processing: filter low-quality cells and genes
## Filter cells: remove cells with extremely high or low number of genes or UMI counts
data = data.values.T # [n_cell, n_gene] after transpose
nGene = (data > 0).sum(axis=1) # number of genes expressed in each cell
Q1 = np.percentile(nGene, 25) # 1st quartile value
Q3 = np.percentile(nGene, 75) # 3rd quartile value
IQR = Q3 - Q1 # interquartile range (IQR)
high_v = Q3 + 3 * IQR; low_v = Q1 - 3 * IQR # high and low thresholds for filtering
index1 = np.intersect1d(np.argwhere(nGene <= high_v), np.argwhere(nGene >= low_v)) # index of cells that pass the filter
nUMI = np.sum(data, axis=1) # total UMI counts in each cell. UMI: unique molecular identifier
Q1 = np.percentile(nUMI, 25) # 1st quartile value
Q3 = np.percentile(nUMI, 75) # 3rd quartile value
IQR = Q3 - Q1 # interquartile range (IQR)
high_v = Q3 + 3 * IQR; low_v = Q1 - 3 * IQR # high and low thresholds for filtering
index2 = np.intersect1d(np.argwhere(nUMI <= high_v), np.argwhere(nUMI >= low_v))
index = np.intersect1d(index1, index2)
data = data[index] # [n_cell, n_gene] after filtering cells
cells = cells[index] # cell names after filtering cells

## Filter genes: remove genes that are not expressed in at least 3 cells
data = data.T # [n_gene, n_cell] after transpose
nCell = (data > 0).sum(axis=1) # number of cells expressing each gene
index = np.argwhere(nCell >= 3).flatten() # index of genes that pass the filter
data = data[index] # [n_gene, n_cell] after filtering genes
genes = genes[index] # gene names after filtering genes
print('After filtering: number of genes is {}, number of cells is {}'.format(len(genes), len(cells)))

# 3\ Save data
pd.DataFrame(data, index=genes, columns=cells).to_csv(save_path, sep='\t') # [n_gene, n_cell]
